# GSoC 2026 — NeuroDyads Pre-Task
## Brain-to-Brain Decoder: Validating Neural Synchrony in Human Conversation

**Submitted by:** Vishal Shankar
**Organization:** ML4SCI
**Mentors:** Dr. Evie Malaia & Dr. Brendan Ames
**Date:** March 2026

---

This notebook is my complete solution to the NeuroDyads pre-task. The task involves preprocessing two simultaneous EEG recordings from a conversational dyad, learning joint low-dimensional embeddings using CEBRA, and interpreting what the embedding geometry reveals about neural synchrony during conversation.

The data consists of two raw 64-channel EEG files (Speaker and Listener) recorded simultaneously at 250 Hz using EGI HydroCel nets. The recordings contain three DIN1 markers that define two conversational segments — one with positive affect and one with negative affect.

**My pipeline:**
- **Part 1**: Segmentation by DIN1 markers, VREF channel removal, ICA artifact removal, power spectrum comparison
- **Part 2**: CEBRA embedding on the joint 128-channel matrix with shuffled control
- **Part 3**: Interpreting the embedding geometry and what the control tells us
- **Part 4**: Honest reflection on the biggest limitation of my specific analysis


## Part 1: Preprocessing

### Step 1 — Load Raw EDF Files and Inspect Annotations

Before doing anything, I need to see the raw file structure — how many channels, what the sampling rate is, and most importantly, where the DIN1 markers fall. The entire analysis depends on correct segmentation: if the two files are not cropped to the exact same time window, the joint matrix will not be time-locked and CEBRA will learn nothing meaningful.


In [11]:
import mne
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load both raw EDF files
raw_speaker = mne.io.read_raw_edf('../data/Speaker.edf', preload=True, verbose=False)
raw_listener = mne.io.read_raw_edf('../data/Listener.edf', preload=True, verbose=False)

print("=== SPEAKER ===")
print(f"Channels: {len(raw_speaker.ch_names)}")
print(f"Sampling frequency: {raw_speaker.info['sfreq']} Hz")
print(f"Duration: {raw_speaker.times[-1]:.1f} seconds")
print(f"Annotations:\n{raw_speaker.annotations}")

print("\n=== LISTENER ===")
print(f"Channels: {len(raw_listener.ch_names)}")
print(f"Sampling frequency: {raw_listener.info['sfreq']} Hz")
print(f"Duration: {raw_listener.times[-1]:.1f} seconds")
print(f"Annotations:\n{raw_listener.annotations}")


=== SPEAKER ===
Channels: 65
Sampling frequency: 250.0 Hz
Duration: 303.0 seconds
Annotations:
<Annotations | 5 segments: DIN1 (3), VBeg (1), VEnd (1)>

=== LISTENER ===
Channels: 65
Sampling frequency: 250.0 Hz
Duration: 303.0 seconds
Annotations:
<Annotations | 3 segments: DIN1 (3)>


### Step 2 — Extract DIN1 Marker Times

Both files have 65 channels and exactly 3 DIN1 markers as expected. Duration matches (303 seconds each) which is a good sign that the recordings are aligned. 

Now I need to extract the exact timestamps of the 3 DIN1 markers to perform the segmentation:
- **Segment 1 (positive affect):** DIN1 marker 1 → DIN1 marker 2
- **Segment 2 (negative affect):** DIN1 marker 3 → end of file


In [14]:
# Extract DIN1 marker onset times from Speaker file
# (Both files have the same markers - using Speaker as reference)

din1_onsets = [ann['onset'] for ann in raw_speaker.annotations if ann['description'] == 'DIN1']

print("DIN1 marker times (seconds):")
for i, t in enumerate(din1_onsets):
    print(f"  Marker {i+1}: {t:.3f}s")

tmin_pos = din1_onsets[0]   # positive affect start
tmax_pos = din1_onsets[1]   # positive affect end
tmin_neg = din1_onsets[2]   # negative affect start
tmax_neg = raw_speaker.times[-1]  # negative affect end = end of file

print(f"\nSegment 1 — Positive affect: {tmin_pos:.3f}s → {tmax_pos:.3f}s  (duration: {tmax_pos - tmin_pos:.1f}s)")
print(f"Segment 2 — Negative affect: {tmin_neg:.3f}s → {tmax_neg:.3f}s  (duration: {tmax_neg - tmin_neg:.1f}s)")


DIN1 marker times (seconds):
  Marker 1: 0.788s
  Marker 2: 148.561s
  Marker 3: 148.835s

Segment 1 — Positive affect: 0.788s → 148.561s  (duration: 147.8s)
Segment 2 — Negative affect: 148.835s → 302.996s  (duration: 154.2s)


### Step 3 — Segment and Remove VREF (Channel 65)

The markers are clean:
- **Positive affect:** 0.79s → 148.56s (~147.8 seconds)
- **Negative affect:** 148.84s → 303.0s (~154.2 seconds)

The gap between marker 2 and 3 is only 0.274 seconds — essentially a seamless transition between the two conversational conditions.

Now I will crop both files to these two segments and remove channel 65 (VREF — the vertex reference electrode). VREF is not an EEG signal; it's the hardware reference and must be dropped before ICA, otherwise it will appear as a dominant component and distort the decomposition.


In [17]:
def segment_and_clean(raw, tmin, tmax, label):
    """Crop to segment and drop VREF channel."""
    r = raw.copy().crop(tmin=tmin, tmax=tmax)
    
    # Drop VREF (channel 65 — index 64)
    vref = [ch for ch in r.ch_names if 'VREF' in ch.upper() or ch == 'E65']
    if vref:
        r.drop_channels(vref)
        dropped = vref[0]
    else:
        dropped = r.ch_names[64]
        r.drop_channels([dropped])
    
    print(f"{label}: {len(r.ch_names)} channels, {r.times[-1]:.1f}s  | Dropped: {dropped}")
    return r

# Segment both files into positive and negative affect
spk_pos = segment_and_clean(raw_speaker, tmin_pos, tmax_pos, "Speaker  — Positive")
spk_neg = segment_and_clean(raw_speaker, tmin_neg, tmax_neg, "Speaker  — Negative")
lst_pos = segment_and_clean(raw_listener, tmin_pos, tmax_pos, "Listener — Positive")
lst_neg = segment_and_clean(raw_listener, tmin_neg, tmax_neg, "Listener — Negative")

print("\nAll 4 segments created — 64 channels each.")


Speaker  — Positive: 64 channels, 147.8s  | Dropped: EEG VREF
Speaker  — Negative: 64 channels, 154.2s  | Dropped: EEG VREF
Listener — Positive: 64 channels, 147.8s  | Dropped: EEG VREF
Listener — Negative: 64 channels, 154.2s  | Dropped: EEG VREF

All 4 segments created — 64 channels each.


### Step 4 — ICA Artifact Removal

With the segments properly cropped and VREF removed, I now apply ICA to remove artifacts — primarily eye blinks (EOG) and muscle noise (EMG).

**Why ICA?** ICA decomposes the multichannel EEG into statistically independent components. Artifacts like eye blinks produce a characteristic frontal topography and slow waveform that are easy to identify. Removing them before feeding data to CEBRA ensures the embedding reflects genuine neural synchrony rather than shared movement artifacts between participants.

**My approach:**
- Apply a 1–40 Hz bandpass filter before ICA fitting (ICA is sensitive to slow drifts)
- Use 20 components — enough to capture dominant sources in 64-channel data without overfitting
- Use MNE's automatic EOG detection to flag blink components
- I will inspect and justify each rejected component


In [20]:
from mne.preprocessing import ICA

def run_ica(raw_seg, n_components=20, label=""):
    """Filter, fit ICA, auto-detect EOG components, return clean raw + ICA object."""
    # Bandpass 1-40 Hz for ICA fitting
    raw_filt = raw_seg.copy().filter(1.0, 40.0, verbose=False)
    
    ica = ICA(n_components=n_components, random_state=42, max_iter='auto', verbose=False)
    ica.fit(raw_filt, verbose=False)
    
    # Auto-detect EOG artifact components using frontal channels
    # EGI HydroCel nets: frontal channels are typically E8, E14, E21, E25
    frontal_chs = [ch for ch in raw_seg.ch_names if ch in ['E8','E14','E21','E25','EEG E8','EEG E14']]
    
    if frontal_chs:
        eog_idx, scores = ica.find_bads_eog(raw_filt, ch_name=frontal_chs[0], verbose=False)
    else:
        eog_idx = []
    
    ica.exclude = eog_idx
    
    # Apply ICA to original (unfiltered) segment
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    
    print(f"{label}: excluded components {eog_idx}")
    return raw_clean, ica

print("Running ICA on all 4 segments...")
spk_pos_clean, ica_spk_pos = run_ica(spk_pos, label="Speaker  — Positive")
spk_neg_clean, ica_spk_neg = run_ica(spk_neg, label="Speaker  — Negative")
lst_pos_clean, ica_lst_pos = run_ica(lst_pos, label="Listener — Positive")
lst_neg_clean, ica_lst_neg = run_ica(lst_neg, label="Listener — Negative")

print("\nICA complete.")


Running ICA on all 4 segments...
Speaker  — Positive: excluded components []
Speaker  — Negative: excluded components []
Listener — Positive: excluded components []
Listener — Negative: excluded components []

ICA complete.


In [22]:
# Auto-detection didn't find EOG channels by name — EGI nets use generic naming.
# Instead, we exclude the top variance components which typically capture 
# eye blink and muscle artifacts in short-segment EEG data.
# We inspect the variance explained by each component and exclude
# the top 2 components from each participant as a conservative approach.

def exclude_top_components(ica, raw_seg, n_exclude=2, label=""):
    """Exclude top n components by variance — standard approach when no EOG channel."""
    # Get explained variance per component
    var = ica.get_explained_variance_ratio(raw_seg, components=list(range(ica.n_components_)))
    
    # Sort by variance descending, take top n
    sorted_comps = sorted(var.items(), key=lambda x: x[1], reverse=True)
    top_comps = [c[0] for c in sorted_comps[:n_exclude]]
    
    ica.exclude = top_comps
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    
    print(f"{label}: excluded components {top_comps} "
          f"(variance: {[round(var[c]*100,1) for c in top_comps]}%)")
    return raw_clean

spk_pos_clean = exclude_top_components(ica_spk_pos, spk_pos, label="Speaker  — Positive")
spk_neg_clean = exclude_top_components(ica_spk_neg, spk_neg, label="Speaker  — Negative")
lst_pos_clean = exclude_top_components(ica_lst_pos, lst_pos, label="Listener — Positive")
lst_neg_clean = exclude_top_components(ica_lst_neg, lst_neg, label="Listener — Negative")

print("\nArtifact removal complete.")


Speaker  — Positive: excluded components ['eeg'] (variance: [-33.6]%)
Speaker  — Negative: excluded components ['eeg'] (variance: [82.8]%)
Listener — Positive: excluded components ['eeg'] (variance: [92.6]%)
Listener — Negative: excluded components ['eeg'] (variance: [94.2]%)

Artifact removal complete.


In [24]:
# Fix: Use component index directly — exclude component 0 and 1
# which are conventionally the highest-variance components in ICA
# These capture the dominant artifacts (eye blinks, cardiac, slow drift)

def apply_ica_fixed(ica, raw_seg, exclude_comps, label=""):
    ica.exclude = exclude_comps
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    print(f"{label}: excluded ICA components {exclude_comps}")
    return raw_clean

# Exclude components 0 and 1 — highest variance components
# Rationale: In continuous resting/conversational EEG, the first ICA components
# almost universally capture eye movement (frontal, low frequency) and 
# cardiac artifacts. This is the standard conservative approach.
spk_pos_clean = apply_ica_fixed(ica_spk_pos, spk_pos, [0, 1], "Speaker  — Positive")
spk_neg_clean = apply_ica_fixed(ica_spk_neg, spk_neg, [0, 1], "Speaker  — Negative")
lst_pos_clean = apply_ica_fixed(ica_lst_pos, lst_pos, [0, 1], "Listener — Positive")
lst_neg_clean = apply_ica_fixed(ica_lst_neg, lst_neg, [0, 1], "Listener — Negative")

print("\nICA artifact removal applied — components 0 & 1 excluded from all segments.")


Speaker  — Positive: excluded ICA components [0, 1]
Speaker  — Negative: excluded ICA components [0, 1]
Listener — Positive: excluded ICA components [0, 1]
Listener — Negative: excluded ICA components [0, 1]

ICA artifact removal applied — components 0 & 1 excluded from all segments.


### Step 5 — Power Spectrum: Before vs After ICA (Required Comparison)

I plot the power spectral density (PSD) of the Speaker's positive affect segment before and after ICA on the same axes. This shows what the artifact removal actually changed in the frequency domain.

In [29]:
import os

# Compute PSD before and after ICA for Speaker — Positive segment
psd_before = spk_pos.compute_psd(method='welch', fmin=0.5, fmax=45, verbose=False)
psd_after  = spk_pos_clean.compute_psd(method='welch', fmin=0.5, fmax=45, verbose=False)

# Average across channels
freqs = psd_before.freqs
power_before = psd_before.get_data().mean(axis=0)
power_after  = psd_after.get_data().mean(axis=0)

# Convert to dB
power_before_db = 10 * np.log10(power_before)
power_after_db  = 10 * np.log10(power_after)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freqs, power_before_db, color='steelblue', linewidth=1.5, label='Before ICA')
ax.plot(freqs, power_after_db,  color='tomato',    linewidth=1.5, label='After ICA')
ax.set_xlabel('Frequency (Hz)', fontsize=12)
ax.set_ylabel('Power (dB)', fontsize=12)
ax.set_title('Speaker — Positive Affect\nPower Spectral Density: Before vs After ICA', fontsize=13)
ax.legend(fontsize=11)
ax.axvspan(0.5, 4, alpha=0.08, color='purple', label='Delta')
ax.axvspan(8, 12, alpha=0.08, color='green',  label='Alpha')
ax.grid(True, alpha=0.3)
plt.tight_layout()

os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/psd_before_after_ica.png', dpi=150)
plt.show()
print("Figure saved to ../figures/psd_before_after_ica.png")


Figure saved to ../figures/psd_before_after_ica.png


### PSD Interpretation

The power spectrum before and after ICA shows the effect of removing ICA components 0 and 1.

**What I observe:**
- The low-frequency region (delta, 0.5–4 Hz) shows a reduction in power after ICA. This is consistent with removing slow eye-movement artifacts, which typically manifest as high-amplitude, low-frequency deflections predominantly in frontal channels.
- The alpha band (8–12 Hz) is largely preserved, which is a good sign — it means we did not remove genuine neural oscillations.
- The 1/f slope (power decreasing with frequency) is maintained in both curves, indicating the ICA did not distort the broadband spectral structure.

**Why I excluded components 0 and 1:**  
Without a dedicated EOG channel in this EGI dataset, I could not use MNE's automatic EOG correlator. In 64-channel continuous EEG, ICA components are sorted by variance explained. The first two components nearly always capture the dominant non-neural sources — eye blinks and slow drift — due to their large amplitude relative to neural signals. This is a well-established heuristic in EEG preprocessing literature and a conservative choice: it removes the most contaminated components while keeping the majority of neural signal intact.

**Uncertainty:** I cannot confirm without visual inspection of the component topographies that components 0 and 1 are purely artifactual. With more time I would plot the ICA topographies and time courses to validate this decision.


---

## Part 2: CEBRA Embedding

### Step 1 — Build Joint T×128 Matrix

CEBRA requires a single joint matrix where both participants' signals are concatenated channel-wise. The first 64 columns are the Speaker, the last 64 are the Listener. Every row represents the same time point — this is what makes it dyadic. If the two participants are not time-locked, CEBRA has no basis to learn brain-to-brain structure.

I Z-normalize each channel independently across time before concatenating. This is important because raw EEG amplitude varies across channels due to electrode impedance differences — normalization ensures no single channel dominates the embedding.

I build this for both the positive and negative affect segments separately, then stack them with labels (0 = positive affect, 1 = negative affect) for CEBRA training.


In [33]:
def build_joint_matrix(raw_a, raw_b):
    """Build Z-normalized joint T×128 matrix from two 64-channel raws."""
    da = raw_a.get_data().T   # (T, 64)
    db = raw_b.get_data().T   # (T, 64)
    
    # Trim to same length (safety measure)
    T = min(len(da), len(db))
    da, db = da[:T], db[:T]
    
    # Z-normalize each channel independently
    da = (da - da.mean(axis=0)) / (da.std(axis=0) + 1e-10)
    db = (db - db.mean(axis=0)) / (db.std(axis=0) + 1e-10)
    
    # Concatenate: Speaker first 64 cols, Listener last 64 cols
    joint = np.concatenate([da, db], axis=1).astype(np.float32)
    return joint

# Build joint matrices for both segments
joint_pos = build_joint_matrix(spk_pos_clean, lst_pos_clean)
joint_neg = build_joint_matrix(spk_neg_clean, lst_neg_clean)

# Labels: 0 = positive affect, 1 = negative affect
labels_pos = np.zeros(len(joint_pos), dtype=int)
labels_neg = np.ones(len(joint_neg), dtype=int)

# Stack both segments
X = np.concatenate([joint_pos, joint_neg], axis=0)
y = np.concatenate([labels_pos, labels_neg], axis=0)

print(f"Positive segment: {joint_pos.shape}  — label 0")
print(f"Negative segment: {joint_neg.shape}  — label 1")
print(f"Full joint matrix: {X.shape}")
print(f"Labels: {np.unique(y, return_counts=True)}")


Positive segment: (36944, 128)  — label 0
Negative segment: (38541, 128)  — label 1
Full joint matrix: (75485, 128)
Labels: (array([0, 1]), array([36944, 38541], dtype=int64))


### Step 2 — Fit CEBRA

CEBRA is a contrastive representation learning method that learns low-dimensional embeddings guided by a behavioural label — in this case the affect condition (0 = positive, 1 = negative). It uses a time-contrastive loss: samples close in time (and sharing the same label) are pulled together in embedding space, while distant or mismatched samples are pushed apart.

I use 3-dimensional output embeddings as required, with 5000 training iterations — sufficient for convergence on this dataset size without over-engineering. The `time_delta` conditional mode is appropriate here as the neural signal is time-continuous.


In [36]:
from cebra import CEBRA

# Fit CEBRA on the full joint matrix with affect labels
model = CEBRA(
    model_architecture='offset10-model',
    batch_size=512,
    learning_rate=3e-4,
    temperature=1.0,
    output_dimension=3,
    max_iterations=5000,
    distance='cosine',
    conditional='time_delta',
    device='cpu',
    verbose=True,
    time_offsets=10,
)

print("Fitting CEBRA... (this takes 2-5 minutes on CPU)")
model.fit(X, y)
print("\nCEBRA training complete.")


Fitting CEBRA... (this takes 2-5 minutes on CPU)


pos: -1.0000 neg:  6.6720 total:  5.6721 temperature:  1.0000: 100%|██████████| 5000/5000 [13:07<00:00,  6.35it/s]


CEBRA training complete.


In [38]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Get embeddings for real data
embedding = model.transform(X)
emb_pos = embedding[:len(joint_pos)]
emb_neg = embedding[len(joint_pos):]

print(f"Embedding shape (full): {embedding.shape}")
print(f"Embedding shape (positive affect): {emb_pos.shape}")
print(f"Embedding shape (negative affect): {emb_neg.shape}")

# KNN decoding accuracy — can a classifier decode affect condition from embedding?
knn = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn, embedding, y, cv=5, scoring='accuracy')
print(f"\nKNN decoding accuracy (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}")

# Goodness of fit score
gof = model.score(X, y)
print(f"CEBRA goodness-of-fit score: {gof:.4f}")


Embedding shape (full): (75485, 3)
Embedding shape (positive affect): (36944, 3)
Embedding shape (negative affect): (38541, 3)

KNN decoding accuracy (5-fold CV): 1.000 ± 0.000


AttributeError: 'CEBRA' object has no attribute 'score'

In [40]:
# CEBRA .score() not available in v0.6.0
# Use training loss as goodness-of-fit proxy instead
# Lower InfoNCE loss = better embedding structure

final_loss = model.state_dict_['loss']
print(f"CEBRA final training loss (InfoNCE): {final_loss:.4f}")
print(f"KNN decoding accuracy (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}")
print("\nInterpretation: KNN accuracy of 1.0 means the two affect conditions")
print("are perfectly linearly separable in the 3D embedding space.")


TypeError: unsupported format string passed to Tensor.__format__

In [42]:
final_loss = float(model.state_dict_['loss'].item())
print(f"CEBRA final training loss (InfoNCE): {final_loss:.4f}")
print(f"KNN decoding accuracy (5-fold CV):   {scores.mean():.3f} ± {scores.std():.3f}")
print("\nKNN accuracy of 1.0 = the two affect conditions are perfectly")
print("separable in the 3D embedding space — strong structure learned.")


RuntimeError: a Tensor with 5000 elements cannot be converted to Scalar

In [44]:
final_loss = float(model.state_dict_['loss'][-1].item())
print(f"CEBRA final training loss (InfoNCE): {final_loss:.4f}")
print(f"KNN decoding accuracy (5-fold CV):   {scores.mean():.3f} ± {scores.std():.3f}")
print("\nKNN accuracy of 1.0 = the two affect conditions are perfectly")
print("separable in the 3D embedding space — strong structure learned.")


CEBRA final training loss (InfoNCE): 5.6721
KNN decoding accuracy (5-fold CV):   1.000 ± 0.000

KNN accuracy of 1.0 = the two affect conditions are perfectly
separable in the 3D embedding space — strong structure learned.


### Step 3 — Shuffled Control

A KNN accuracy of 1.0 on real data is striking, but we need to verify it reflects genuine neural structure and not an artifact of temporal autocorrelation in the data. The shuffled control breaks the time-locking between Speaker and Listener by randomly permuting one participant's data. If CEBRA still learns a good embedding on shuffled data, it means the structure comes from within-participant signal, not the dyadic relationship.


In [47]:
# Shuffled control — break time-locking between Speaker and Listener
# Shuffle only the Listener half (last 64 columns) within each segment

np.random.seed(42)

def shuffle_listener(joint_matrix):
    """Shuffle the Listener (last 64 channels) time axis independently."""
    shuffled = joint_matrix.copy()
    idx = np.random.permutation(len(shuffled))
    shuffled[:, 64:] = shuffled[idx, 64:]  # shuffle Listener rows only
    return shuffled

X_shuffled = np.concatenate([
    shuffle_listener(joint_pos),
    shuffle_listener(joint_neg)
], axis=0)

# Train CEBRA on shuffled data
model_ctrl = CEBRA(
    model_architecture='offset10-model',
    batch_size=512,
    learning_rate=3e-4,
    temperature=1.0,
    output_dimension=3,
    max_iterations=5000,
    distance='cosine',
    conditional='time_delta',
    device='cpu',
    verbose=True,
    time_offsets=10,
)

print("Fitting CEBRA on shuffled control...")
model_ctrl.fit(X_shuffled, y)
print("\nControl training complete.")


Fitting CEBRA on shuffled control...


pos: -0.9999 neg:  6.6724 total:  5.6725 temperature:  1.0000: 100%|██████████| 5000/5000 [39:36<00:00,  2.10it/s]  


Control training complete.


In [49]:
# Get control embeddings and compare KNN accuracy
emb_ctrl = model_ctrl.transform(X_shuffled)

knn_ctrl = KNeighborsClassifier(n_neighbors=5)
scores_ctrl = cross_val_score(knn_ctrl, emb_ctrl, y, cv=5, scoring='accuracy')

ctrl_loss = float(model_ctrl.state_dict_['loss'][-1].item())

print("=" * 50)
print("        RESULTS SUMMARY")
print("=" * 50)
print(f"Real data:")
print(f"  KNN accuracy:      {scores.mean():.3f} ± {scores.std():.3f}")
print(f"  InfoNCE loss:      {final_loss:.4f}")
print()
print(f"Shuffled control:")
print(f"  KNN accuracy:      {scores_ctrl.mean():.3f} ± {scores_ctrl.std():.3f}")
print(f"  InfoNCE loss:      {ctrl_loss:.4f}")
print("=" * 50)


        RESULTS SUMMARY
Real data:
  KNN accuracy:      1.000 ± 0.000
  InfoNCE loss:      5.6721

Shuffled control:
  KNN accuracy:      1.000 ± 0.000
  InfoNCE loss:      5.6725


In [51]:
# Visualize real vs shuffled embeddings side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue' if label == 0 else 'tomato' for label in y]

# Real embedding
axes[0].scatter(embedding[:, 0], embedding[:, 1], c=colors, alpha=0.02, s=1)
axes[0].set_title('Real Data Embedding\n(blue=positive affect, red=negative affect)', fontsize=11)
axes[0].set_xlabel('Dim 1')
axes[0].set_ylabel('Dim 2')

# Shuffled control embedding  
axes[1].scatter(emb_ctrl[:, 0], emb_ctrl[:, 1], c=colors, alpha=0.02, s=1)
axes[1].set_title('Shuffled Control Embedding\n(Listener time-axis permuted)', fontsize=11)
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')

plt.tight_layout()
plt.savefig('../figures/embedding_real_vs_shuffled.png', dpi=150)
plt.show()
print("Figure saved.")


Figure saved.


In [ ]:
cd ~/gsoc2026-neurodyads
git add notebooks/neurodyads_pretask.ipynb figures/embedding_real_vs_shuffled.png
git commit -m "Part 2 complete — CEBRA embedding, KNN accuracy, shuffled control"
git push
